# NYC Yellow Taxi 2026 — Medallion Architecture Analysis

**Dataset:** NYC Taxi and Limousine Commission (TLC) — Yellow Taxi Trip Records, January–April 2026  
**Source:** https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page  
**Architecture:** Bronze → Silver → Gold (Medallion / Lakehouse pattern)  
**Catalog:** taxi | Schemas: taxi_bronze · taxi_silver · taxi_gold

## Project Overview
This notebook implements a full medallion architecture pipeline on NYC Yellow Taxi data:
- **Bronze** — raw ingestion of Parquet files and the TLC zone lookup table, stored as Delta tables with no transformations
- **Silver** — cleaned, validated, and enriched data: invalid rows removed, derived columns added (trip duration, speed, tip %, time features, labels)
- **Gold** — one Delta table per analytical question, ready for dashboarding

## Data Dictionary References
All column definitions, RatecodeID values, VendorIDs, and payment types referenced throughout this notebook are sourced from the official NYC TLC data dictionary:  
https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf

In [0]:
df_br = spark.read.parquet("/Volumes/taxi/taxi_bronze/yellow_taxi_trip_data")
# This reads all 5 monthly Parquet files at once from the Volume
print(f"Total rows: {df_br.count():,}")
print(f"Columns: {len(df_br.columns)}")

## Bronze Layer

Raw ingestion only. No transformations, no filtering, no renaming.  
Bronze is the source of truth — if Silver or Gold need to be rebuilt, Bronze always has the complete original data.

Two tables are loaded:
- `taxi.taxi_bronze.yellow_taxi_trip_raw_data` — raw trip records from 5 monthly Parquet files (Jan–Apr 2026)
- `taxi.taxi_bronze.yellow_taxi_lookup_raw_data` — TLC taxi zone lookup table (265 rows mapping LocationID to Zone, Borough, and service_zone)

In [0]:
df_lookup = spark.read.option("header", True).option("inferSchema", True).csv("/Volumes/taxi/taxi_bronze/taxi_zone_lookup")
print(f"Total rows: {df_lookup.count():,}")
print(f"Columns: {len(df_lookup.columns)}")

In [0]:
# Writing all 5 monthly Parquet files as a single Delta table in Bronze.
# Spark merges the files automatically since they share the same schema.
# Delta format adds ACID transactions and time-travel on top of Parquet.
(df_br
.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable("taxi.taxi_bronze.yellow_taxi_trip_raw_data"))

In [0]:
# Zone lookup reference table — 265 rows mapping LocationID (integer) to
# Zone name, Borough, and service_zone. 
# Used in Gold analyses to convert PULocationID and DOLocationID integers
# into readable zone and borough names.
# Stored in Bronze alongside trip data following standard medallion convention
# where all raw ingested data lands in Bronze regardless of type.
(df_lookup
.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable("taxi.taxi_bronze.yellow_taxi_lookup_raw_data"))

In [0]:
df_taxi_bronze = spark.read.table("taxi.taxi_bronze.yellow_taxi_trip_raw_data")
df_lookup_bronze = spark.read.table("taxi.taxi_bronze.yellow_taxi_lookup_raw_data")


In [0]:
df_taxi_bronze.printSchema()
df_taxi_bronze.show(10, truncate=False)
df_lookup_bronze.printSchema()
df_lookup_bronze.show(10, truncate=False)

## Bronze Data Quality Checks

Before building Silver, we measure how many rows contain invalid values across key columns.
These counts inform the Silver filtering logic — values are not removed here (Bronze stays untouched).
Results are noted inline as comments for reference.

### Bronze Exploration: Row Counts

In [0]:
%sql  
SELECT 'trip_data'   AS table_name, COUNT(*) AS row_count FROM taxi.taxi_bronze.yellow_taxi_trip_raw_data
UNION ALL
SELECT 'zone_lookup' AS table_name, COUNT(*) AS row_count FROM taxi.taxi_bronze.yellow_taxi_lookup_raw_data;

### Bronze Exploration: Distinct VendorIDs
Expected values per TLC data dictionary: 1, 2, 6, 7.
Any other values are invalid and will be handled in Silver.

In [0]:
%sql
SELECT VendorID, COUNT(*) AS trip_count
FROM taxi.taxi_bronze.yellow_taxi_trip_raw_data
GROUP BY VendorID
ORDER BY VendorID;

In [0]:
%sql
SELECT * 
FROM taxi.taxi_bronze.yellow_taxi_trip_raw_data
WHERE VendorID = 7
  AND passenger_count BETWEEN 1 AND 6;

### Bronze Exploration: Lookup Table
Spot check: LocationID 132 = JFK Airport — confirms lookup table loaded correctly

In [0]:
%sql
SELECT * FROM taxi.taxi_bronze.yellow_taxi_lookup_raw_data WHERE LocationID = 132;

## Silver Layer

Cleaned, validated, and enriched data built from Bronze.  
Silver applies all data quality decisions identified in the Bronze exploration above.

**What is removed:**
- Trips where `trip_distance <= 0`, `fare_amount <= 0`, or `total_amount <= 0`
- Trips where `passenger_count` is 0 or greater than 6
- Trips where `tpep_dropoff_datetime <= tpep_pickup_datetime` (impossible timestamps)
- Trips with `tpep_pickup_datetime` before 2026-01-01 or after 2026-12-31 (metering clock errors)

**What is added:**
- `vendorid_label`, `rate_type`, `payment_type_label` — human-readable labels replacing numeric codes
- `pickup_hour`, `pickup_day_of_week`, `pickup_month` — extracted time dimensions
- `dropoff_hour`, `dropoff_day_of_week`, `dropoff_month` — extracted time dimensions
- `trip_duration_mins` — calculated from pickup to dropoff timestamps
- `avg_speed_mph` — calculated from distance and duration
- `tip_pct` — tip as % of fare, credit card payments only (cash tips not recorded)
- `trip_type` — airport vs standard, based on RatecodeID and Airport_fee
- `congestion_zone_flag` — congestion zone vs standard, based on cbd_congestion_fee

In [0]:
%sql
-- Dropping all the duplicates.
CREATE OR REPLACE TABLE taxi.taxi_silver.yellow_taxi_trips_cleaned AS
WITH dedupl AS (
SELECT *, ROW_NUMBER() OVER(PARTITION BY tpep_pickup_datetime, tpep_dropoff_datetime, PULocationID, DOLocationID, total_amount ORDER BY tpep_pickup_datetime) AS rn 
FROM taxi.taxi_bronze.yellow_taxi_trip_raw_data
)
SELECT * EXCEPT(rn) 
FROM dedupl
WHERE rn=1;

In [0]:
%sql
SELECT COUNT(*) 
FROM taxi.taxi_silver.yellow_taxi_trips_cleaned
WHERE passenger_count <= 0 OR passenger_count > 6; -- 64885

SELECT COUNT(*) 
FROM taxi.taxi_silver.yellow_taxi_trips_cleaned
WHERE trip_distance <= 0; -- 576744

SELECT COUNT(*) 
FROM taxi.taxi_silver.yellow_taxi_trips_cleaned
WHERE fare_amount <= 0 OR total_amount <= 0; -- 129194

SELECT COUNT(*)
FROM taxi.taxi_silver.yellow_taxi_trips_cleaned
WHERE  tpep_pickup_datetime >= tpep_dropoff_datetime; -- 236375

SELECT DISTINCT RatecodeID, COUNT(*) 
FROM taxi.taxi_silver.yellow_taxi_trips_cleaned
GROUP BY RatecodeID
ORDER BY RatecodeID; -- RatecodeID null (4812280 rows) and 99 (637889 rows).

In [0]:
%sql
CREATE OR REPLACE TABLE taxi.taxi_silver.yellow_taxi_trips_cleaned AS
SELECT * EXCEPT (store_and_fwd_flag, mta_tax, improvement_surcharge), 
        CASE
        WHEN VendorID = 1 THEN 'Creative Mobile Technologies, LLC' 
        WHEN VendorID = 2 THEN 'Curb Mobility, LLC ' 
        WHEN VendorID = 6 THEN 'Myle Technologies Inc' 
        WHEN VendorID = 7 THEN 'Helix'
        ELSE 'unknown_vendor'
        END AS vendorid_label,
    -- Label rate type including NULL handling
        CASE 
        WHEN RatecodeID IS NULL OR RatecodeID = 99  THEN 'unknown'
        WHEN RatecodeID = 1 THEN 'standard'
        WHEN RatecodeID = 2 THEN 'JFK'
        WHEN RatecodeID = 3 THEN 'Newark'
        WHEN RatecodeID = 4 THEN 'Nassau/Westchester'
        WHEN RatecodeID = 5 THEN 'negotiated'
        WHEN RatecodeID = 6 THEN 'group_ride'
    END AS rate_type,
        CASE
        WHEN payment_type = 0 THEN 'flex fare trip'
        WHEN payment_type = 1 THEN 'credit_card'
        WHEN payment_type = 2 THEN 'cash'
        WHEN payment_type = 3 THEN 'no_charge'
        WHEN payment_type = 4 THEN 'dispute'
        WHEN payment_type = 5 THEN 'unknown'
        ELSE 'voided'
        END AS payment_type_label,
    -- Derived time columns
    EXTRACT(HOUR FROM tpep_pickup_datetime)        AS pickup_hour,
    EXTRACT(DAYOFWEEK FROM tpep_pickup_datetime)   AS pickup_day_of_week,
    DATE_FORMAT(tpep_pickup_datetime, 'yyyy-MM')   AS pickup_month,
    EXTRACT(HOUR FROM tpep_dropoff_datetime)        AS dropoff_hour,
    EXTRACT(DAYOFWEEK FROM tpep_dropoff_datetime)   AS dropoff_day_of_week,
    DATE_FORMAT(tpep_dropoff_datetime, 'yyyy-MM')   AS dropoff_month,
    
    -- Trip duration in minutes
    ROUND((UNIX_TIMESTAMP(tpep_dropoff_datetime) - 
           UNIX_TIMESTAMP(tpep_pickup_datetime)) / 60, 2) AS trip_duration_mins,
    
    -- Speed in mph
    ROUND(trip_distance / 
          NULLIF((UNIX_TIMESTAMP(tpep_dropoff_datetime) - 
                  UNIX_TIMESTAMP(tpep_pickup_datetime)) / 3600, 0), 2) AS avg_speed_mph,
    
    -- Tip percentage (card payments only — cash tips not recorded)
    CASE 
        WHEN payment_type = 1
        THEN ROUND(tip_amount / NULLIF(fare_amount,0) * 100, 2)
        ELSE NULL 
    END AS tip_pct,
    
    -- Trip type flags
    CASE WHEN RatecodeID IN (2, 3) OR Airport_fee > 0 
         THEN 'airport'
         ELSE 'standard' 
    END AS trip_type,
    
    CASE WHEN cbd_congestion_fee > 0 
         THEN 'congestion_zone'
         ELSE 'standard' 
    END AS congestion_zone_flag

FROM taxi.taxi_silver.yellow_taxi_trips_cleaned
WHERE fare_amount >= 0 
  AND total_amount >= 0 
  AND trip_distance >= 0
  AND TIMESTAMPDIFF(SECOND, tpep_pickup_datetime, tpep_dropoff_datetime) / 60 BETWEEN 5 AND 200
  AND passenger_count BETWEEN 1 AND 6
  AND tpep_pickup_datetime >= '2026-01-01'
  AND tpep_dropoff_datetime > tpep_pickup_datetime;

/*
From the official NYC TLC data dictionary for the VendorID of yellow taxis:
1 = Creative Mobile Technologies, LLC 
2 = Curb Mobility, LLC 
6 = Myle Technologies Inc 
7 = Helix
*/

/*
From the official NYC TLC data dictionary for the RatecodeID of yellow taxis:
1 = Standard rate 
2 = JFK 
3 = Newark 
4 = Nassau or Westchester 
5 = Negotiated fare 
6 = Group ride 
99 = Null/unknown 
*/

/*
From the official NYC TLC data dictionary for the payment_types of yellow taxis:
0 =  = Flex Fare trip 
1 = Credit card
2 = Cash
3 =	No charge
4 =	Dispute
5 =	Unknown
6 =	Voided trip
Note: Cash TIPS are not electronically recorded.
Also the payment types seen in this dataset are from 0 to 4.
*/

/*
NOTE: mta_tax and improvement_surcharge were excluded from Silver.
Despite being documented as fixed values ($0.50 and $0.30 respectively),
both columns contain highly inconsistent values in this dataset
(mta_tax ranges from -$0.50 to $10.50, improvement_surcharge from -$1 to $2.50)
suggesting unreliable recording by vendor metering systems.
As a result total_amount will not reconcile with the sum of remaining components.
total_amount will be used for revenue analysis and individual components for breakdown only.
*/

In [0]:
%sql
-- Reordering columns for readability only — no data changes.
-- Labels and derived columns are placed next to the source columns they describe.
ALTER TABLE taxi.taxi_silver.yellow_taxi_trips_cleaned
ALTER COLUMN vendorid_label
AFTER VendorID;

ALTER TABLE taxi.taxi_silver.yellow_taxi_trips_cleaned
ALTER COLUMN payment_type_label
AFTER payment_type;

ALTER TABLE taxi.taxi_silver.yellow_taxi_trips_cleaned
ALTER COLUMN rate_type
AFTER RatecodeID;

ALTER TABLE taxi.taxi_silver.yellow_taxi_trips_cleaned
ALTER COLUMN tip_pct
AFTER tip_amount;

ALTER TABLE taxi.taxi_silver.yellow_taxi_trips_cleaned
ALTER COLUMN trip_duration_mins
AFTER tpep_dropoff_datetime;

ALTER TABLE taxi.taxi_silver.yellow_taxi_trips_cleaned
ALTER COLUMN avg_speed_mph
AFTER trip_distance;

ALTER TABLE taxi.taxi_silver.yellow_taxi_trips_cleaned
ALTER COLUMN pickup_month
AFTER tpep_pickup_datetime; 

ALTER TABLE taxi.taxi_silver.yellow_taxi_trips_cleaned
ALTER COLUMN  pickup_day_of_week
AFTER pickup_month; 

ALTER TABLE taxi.taxi_silver.yellow_taxi_trips_cleaned
ALTER COLUMN pickup_hour
AFTER pickup_day_of_week; 

ALTER TABLE taxi.taxi_silver.yellow_taxi_trips_cleaned
ALTER COLUMN dropoff_month
AFTER tpep_dropoff_datetime; 

ALTER TABLE taxi.taxi_silver.yellow_taxi_trips_cleaned
ALTER COLUMN  dropoff_day_of_week
AFTER dropoff_month; 

ALTER TABLE taxi.taxi_silver.yellow_taxi_trips_cleaned
ALTER COLUMN dropoff_hour
AFTER dropoff_day_of_week; 

### Silver Verification
Confirm row count, schema, and that RatecodeID values are now handled correctly.

In [0]:
%sql
SELECT * 
FROM taxi.taxi_silver.yellow_taxi_trips_cleaned
LIMIT 10;
    
SELECT COUNT(*) 
FROM taxi.taxi_silver.yellow_taxi_trips_cleaned;

SELECT DISTINCT RatecodeID 
FROM taxi.taxi_silver.yellow_taxi_trips_cleaned
ORDER BY RatecodeID;

### Silver Quality Checks

Two additional checks beyond schema and row count:

**Trip duration cap:** Verify no trips exceed 200 minutes remain in Silver.
A 200-minute cap was applied during Silver cleaning to remove meter errors where
the taxi meter was left running for hours without a legitimate trip occurring.
The single June 2026 trip that survived all other filters had a duration of
1,416 minutes (nearly 24 hours) and was correctly excluded by this cap.

**June 2026 exclusion:** Verify no June 2026 trips remain in Silver.
The source Parquet file for June 2026 contained only one trip record —
almost certainly an incomplete data export rather than a real month of data.
Including a single trip as a full month would distort every monthly average,
MoM calculation, and trend line across the entire analysis.
June 2026 was therefore excluded from Silver via the timestamp filter.

In [0]:
%sql
SELECT COUNT(*) 
FROM taxi.taxi_silver.yellow_taxi_trips_cleaned
WHERE trip_duration_mins > 200; -- Expected: 0. Cap applied in Silver WHERE clause to remove meter errors. 

SELECT COUNT(*), pickup_month
FROM taxi.taxi_silver.yellow_taxi_trips_cleaned
GROUP BY pickup_month
ORDER BY pickup_month; -- Expected: no 2026-06 row. June excluded — source file contained only 1 trip.

**Trip & Fare Analysis**

## Gold Layer

One Delta table per analytical question, built from Silver.  
Gold tables are the serving layer — pre-computed and ready for dashboarding.

**Tables created:**
1. `trip_duration_speed_analysis` — fare efficiency by trip duration bucket
2. `fare_components_breakdown` — fare breakdown by payment method
3. `surge_detection_general` — all hour × day combinations with revenue metrics
3b. `surge_detection` — top 5 highest-revenue hour × day combinations
4. `top_pickup_dropoff_zones` — zones appearing in top 10 for both pickup and dropoff
5. `zone_to_zone_frequency` — top 20 most frequent corridors
5b. `zone_to_zone_profitability` — top 20 most profitable corridors by fare per minute
6. `trip_type_analysis` — airport vs standard trip comparison
6b. `trip_type_in_depth_analysis` — top 15 pickup zones for airport trips
7. `congestion_impact_by_corridor` — congestion zone vs standard by borough corridor
7b. `congestion_impact_monthly` — monthly trend by congestion flag
8. `vendor_performance` — vendor comparison across all key metrics
9. `monthly_trend_analysis` — month-over-month growth across all key metrics

1. Trip duration and speed analysis

Find the distribution of trips by duration bucket (5-15 min, 15-30 min, 30-45 min, 45+ min) and identify which buckets generate the most revenue per minute.

In [0]:
%sql
CREATE OR REPLACE TABLE taxi.taxi_gold.trip_duration_speed_analysis AS 
WITH trip_buckets  AS (
SELECT trip_duration_mins, avg_speed_mph, fare_amount, ROUND(fare_amount/NULLIF(trip_duration_mins,0),2) AS fare_per_min,
CASE WHEN trip_duration_mins BETWEEN 5 AND 15 THEN '5-15 min'
WHEN trip_duration_mins BETWEEN 15 AND 30 THEN '15-30 min'
WHEN trip_duration_mins BETWEEN 30 AND 45 THEN '30-45 min'
ELSE '45+ min'
END AS trip_duration_bucket
FROM taxi.taxi_silver.yellow_taxi_trips_cleaned
),
bucket_summary  AS (
SELECT trip_duration_bucket, COUNT(trip_duration_mins) AS total_trips, 
    ROUND(AVG(fare_per_min), 2)                        AS avg_fare_per_min,    -- weighted: efficiency regardless of volume
    ROUND(SUM(fare_amount), 2)                         AS total_revenue        -- absolute: contribution by volume
FROM trip_buckets 
GROUP BY trip_duration_bucket
)
SELECT
    trip_duration_bucket,
    total_trips,
    -- share of total trips
    ROUND(total_trips / SUM(total_trips) OVER() * 100, 2)       AS trip_share_pct,
    avg_fare_per_min,
    total_revenue,
    -- share of total revenue
    ROUND(total_revenue / SUM(total_revenue) OVER() * 100, 2)   AS revenue_share_pct
FROM bucket_summary
ORDER BY trip_duration_bucket;

/*
avg_fare_per_min:	Which duration is most efficient for the driver?
total_revenue:  	Which duration generates the most absolute revenue?
trip_share_pct: 	How common is each duration?
revenue_share_pct:	Which duration dominates revenue?
*/

2. Fare components breakdown

For each payment type, calculate the average breakdown of total_amount into its components: fare_amount, tip_amount, tolls_amount, congestion_surcharge, airport_fee, cbd_congestion_fee, extra.

In [0]:
%sql
CREATE OR REPLACE TABLE taxi.taxi_gold.fare_components_breakdown AS 
WITH payment_totals   AS (
SELECT payment_type,
        payment_type_label, 
        COUNT(*) AS total_trips,
        SUM(fare_amount) AS total_fare_amount, 
        SUM(tip_amount) AS total_tip_amount, 
        SUM(tolls_amount) AS total_tolls_amount, 
        SUM(congestion_surcharge) AS total_congestion_surcharge, 
        SUM(airport_fee) AS total_airport_fee, 
        SUM(cbd_congestion_fee) AS total_cbd_congestion_fee, 
        SUM(extra) AS total_extra, 
        SUM(total_amount) AS total_amount
FROM taxi.taxi_silver.yellow_taxi_trips_cleaned
GROUP BY payment_type, payment_type_label
)
SELECT  payment_type,
        payment_type_label,
        total_trips,
        ROUND(total_fare_amount/total_amount *100,2) AS pct_fare,
        -- Tip rate as % of fare TOTAL (more meaningful than % of total)
        ROUND(total_tip_amount / total_fare_amount * 100, 2)     AS tip_rate_pct,
        ROUND(total_tip_amount/total_amount *100,2) AS pct_tip,
        ROUND(total_tolls_amount/total_amount *100,2) AS pct_tolls,
        ROUND(total_congestion_surcharge/total_amount *100,2) AS pct_congestion_surcharge, 
        ROUND(total_airport_fee/total_amount *100,2) AS pct_airport, 
        ROUND(total_cbd_congestion_fee/total_amount *100,2) AS pct_cbd_congestion_fee, 
        ROUND(total_extra/total_amount *100,2) AS pct_extra, 
        ROUND(total_amount,2) AS total_amount
FROM payment_totals 
ORDER BY payment_type;

3. Surge detection — revenue by hour and day

For each hour of the day (0-23) and day of week, calculate average fare, total trip count, and average tip percentage. 
Identify the top 5 hour+day combinations by total revenue — the effective surge windows where demand peaks.

In [0]:
%sql
CREATE OR REPLACE TABLE taxi.taxi_gold.surge_detection_general AS 
WITH hourly_stats AS (
SELECT pickup_day_of_week, pickup_hour,
       COUNT(*)                   AS trip_count, 
       ROUND(AVG(fare_amount),2)  AS avg_fare,
       -- Tip rate on credit card trips only — cash tips not electronically recorded
       -- Calculated as tip / fare (not tip / total) which is the standard industry metric
       ROUND(AVG(CASE WHEN payment_type = 1 THEN tip_amount/NULLIF(fare_amount, 0) END) *100,2)   AS avg_tip_prc,
       ROUND(AVG(total_amount),2) AS avg_total_amount,
       ROUND(SUM(total_amount),2) AS total_revenue
FROM taxi.taxi_silver.yellow_taxi_trips_cleaned
GROUP BY pickup_hour, pickup_day_of_week
)
SELECT *
FROM hourly_stats
ORDER BY pickup_day_of_week, pickup_hour;


In [0]:
%sql
CREATE OR REPLACE TABLE taxi.taxi_gold.surge_detection AS 
WITH hourly_stats AS (
SELECT pickup_hour, pickup_day_of_week,
       COUNT(*)                   AS trip_count, 
       ROUND(AVG(fare_amount),2)  AS avg_fare,
       -- Tip rate on credit card trips only — cash tips not electronically recorded
       -- Calculated as tip / fare (not tip / total) which is the standard industry metric
       ROUND(AVG(CASE WHEN payment_type = 1 THEN tip_amount/NULLIF(fare_amount, 0) END) *100,2)   AS avg_tip_prc,
       ROUND(AVG(total_amount),2) AS avg_total_amount,
       ROUND(SUM(total_amount),2) AS total_revenue
FROM taxi.taxi_silver.yellow_taxi_trips_cleaned
GROUP BY pickup_hour, pickup_day_of_week
),
ranked AS (
    SELECT *,
           DENSE_RANK() OVER (ORDER BY avg_total_amount DESC) AS revenue_rank
    FROM hourly_stats
)
SELECT *
FROM ranked
WHERE revenue_rank <= 5
ORDER BY revenue_rank;

/*
Note: Sunday is consider as the first day of the week.
*/

**Geographic Analysis**

4. Top pickup and dropoff zones

Rank all zones by total trips as pickup location and separately as dropoff location.

Find zones that appear in top 10 for both — the busiest interchange zones in NYC.

In [0]:
%sql
CREATE OR REPLACE TABLE taxi.taxi_gold.top_pickup_dropoff_zones AS
WITH pickup AS (
SELECT a.PULocationID, b.Zone AS pickup_zone, b.Borough AS pickup_borough
FROM taxi.taxi_silver.yellow_taxi_trips_cleaned AS a
JOIN taxi.taxi_bronze.yellow_taxi_lookup_raw_data AS b
ON a.PULocationID = b.LocationID
),
dropoff AS (
SELECT a.DOLocationID, b.Zone AS dropoff_zone, b.Borough AS dropoff_borough
FROM taxi.taxi_silver.yellow_taxi_trips_cleaned AS a
JOIN taxi.taxi_bronze.yellow_taxi_lookup_raw_data AS b
ON a.DOLocationID = b.LocationID
),
pickup_trips AS (
SELECT *, COUNT(*) AS pickup_count
FROM pickup
GROUP BY PULocationID, pickup_zone, pickup_borough
),
dropoff_trips AS (
SELECT *, COUNT(*) AS dropoff_count
FROM dropoff
GROUP BY DOLocationID, dropoff_zone, dropoff_borough
),
combined AS (
SELECT a.pickup_zone AS Area, a.pickup_borough AS Borough, a.pickup_count, b.dropoff_count
FROM pickup_trips AS a
JOIN dropoff_trips AS b
ON a.PULocationID = b.DOLocationID
),
rank_cte AS (
SELECT Area, Borough,
       pickup_count, DENSE_RANK() OVER (ORDER BY pickup_count DESC) AS pickup_rank,
       dropoff_count, DENSE_RANK() OVER (ORDER BY dropoff_count DESC) AS dropoff_rank
FROM combined
)
SELECT Area, Borough, pickup_count, pickup_rank, dropoff_count, dropoff_rank
FROM rank_cte
WHERE pickup_rank <= 10 AND dropoff_rank <= 10
ORDER BY pickup_rank;

-- Only zones appearing in top 10 for both pickup and dropoff

5. Most common pickup → dropoff corridors

Find the top 20 most frequent and top 20 most profitable zone-to-zone corridors (PULocationID → DOLocationID pairs). For each corridor calculate average fare, average tip, and average duration. Which corridors are most profitable per minute?

### Corridor Minimum Trip Count Threshold Analysis

Before building the profitability Gold table, we determine the appropriate
minimum trip count per corridor. A corridor with only 1-2 trips produces
statistically meaningless fare_per_min values — a single $300 fare on a
5-minute trip gives fare_per_min = 60, which dominates the profitability
ranking despite being an outlier rather than a genuine pattern.

This query counts how many corridors survive at different thresholds (10, 20, 50, 100 trips).
The chosen threshold is applied via HAVING COUNT(*) >= N in the Gold table query.
Higher thresholds produce more reliable averages but exclude low-frequency corridors
that may still be analytically interesting.

In [0]:
%sql
SELECT 
    COUNT(CASE WHEN trip_count >= 10  THEN 1 END) AS corridors_10_plus,     -- lenient: removes single-trip noise
    COUNT(CASE WHEN trip_count >= 20 THEN 1 END) AS corridors_20_plus,      -- moderate
    COUNT(CASE WHEN trip_count >= 50  THEN 1 END) AS corridors_50_plus,     -- recommended balance
    COUNT(CASE WHEN trip_count >= 100 THEN 1 END) AS corridors_100_plus,    -- strict: highest reliability
    COUNT(*) AS total_corridors
FROM (
    SELECT PULocationID, DOLocationID, COUNT(*) AS trip_count
    FROM taxi.taxi_silver.yellow_taxi_trips_cleaned
    GROUP BY PULocationID, DOLocationID
);

In [0]:
%sql
CREATE OR REPLACE TABLE taxi.taxi_gold.zone_to_zone_frequency AS
WITH corridor_stats AS (
SELECT PULocationID, DOLocationID, COUNT(*) AS trip_count,
       ROUND(AVG(fare_amount),2) AS avg_fare,
       -- avg_tips: credit card tips only (payment_type = 1) — cash tips not recorded
       ROUND(AVG(CASE WHEN payment_type =1 THEN tip_amount END),2) AS avg_tips,
       ROUND(AVG(trip_duration_mins),2) AS avg_trip_duration,
       ROUND(AVG(fare_amount / NULLIF(trip_duration_mins,0)), 2) AS fare_per_min
FROM taxi.taxi_silver.yellow_taxi_trips_cleaned
GROUP BY PULocationID, DOLocationID
HAVING COUNT(*) >= 20
),
corridor_zones AS (
SELECT b.Zone AS pickup_zone, b.Borough AS pickup_borough, c.Zone AS dropoff_zone, c.Borough AS dropoff_borough, a.trip_count, a.avg_fare, a.avg_tips,
       a.avg_trip_duration, a.fare_per_min,
       DENSE_RANK() OVER(ORDER BY trip_count DESC) AS frequency_rank,
       DENSE_RANK() OVER (ORDER BY fare_per_min DESC) AS profitability_rank
FROM corridor_stats AS a
JOIN taxi.taxi_bronze.yellow_taxi_lookup_raw_data AS b
ON a.PULocationID = b.LocationID
JOIN taxi.taxi_bronze.yellow_taxi_lookup_raw_data AS c
ON a.DOLocationID = c.LocationID
WHERE c.Borough NOT IN ('Unknown', 'N/A') AND b.Borough NOT IN ('Unknown', 'N/A')
)
SELECT *
FROM corridor_zones
WHERE frequency_rank <= 20
ORDER BY frequency_rank;

In [0]:
%sql
CREATE OR REPLACE TABLE taxi.taxi_gold.zone_to_zone_profitability AS
WITH corridor_stats AS (
SELECT PULocationID, DOLocationID, COUNT(*) AS trip_count,
       ROUND(AVG(fare_amount),2) AS avg_fare,
       -- avg_tips: credit card tips only (payment_type = 1) — cash tips not recorded
       ROUND(AVG(CASE WHEN payment_type =1 THEN tip_amount END),2) AS avg_tips,
       ROUND(AVG(trip_duration_mins),2) AS avg_trip_duration,
       ROUND(AVG(fare_amount / NULLIF(trip_duration_mins,0)), 2) AS fare_per_min
FROM taxi.taxi_silver.yellow_taxi_trips_cleaned
WHERE trip_duration_mins <=200 
GROUP BY PULocationID, DOLocationID
HAVING COUNT(*) >= 20
),
corridor_zones AS (
SELECT b.Zone AS pickup_zone, b.Borough AS pickup_borough, c.Zone AS dropoff_zone, c.Borough AS dropoff_borough, a.trip_count, a.avg_fare, a.avg_tips,
       a.avg_trip_duration, a.fare_per_min,
       DENSE_RANK() OVER(ORDER BY trip_count DESC) AS frequency_rank,
       DENSE_RANK() OVER (ORDER BY fare_per_min DESC) AS profitability_rank
FROM corridor_stats AS a
JOIN taxi.taxi_bronze.yellow_taxi_lookup_raw_data AS b
ON a.PULocationID = b.LocationID
JOIN taxi.taxi_bronze.yellow_taxi_lookup_raw_data AS c
ON a.DOLocationID = c.LocationID
WHERE c.Borough NOT IN ('Unknown', 'N/A') AND b.Borough NOT IN ('Unknown', 'N/A')
)
SELECT *
FROM corridor_zones
WHERE profitability_rank <= 20
ORDER BY profitability_rank;

6. Airport trip analysis

Isolate airport trips using RatecodeID (2 = JFK, 3 = Newark) and Airport_fee > 0. Compare airport trips vs non-airport trips: average fare, average tip percentage, average distance, most common pickup zones for airport-bound trips.

In [0]:
%sql
CREATE OR REPLACE TABLE taxi.taxi_gold.trip_type_analysis AS 
WITH trip_type_stats AS (
SELECT trip_type,
       COUNT(*) AS trip_count, 
       ROUND(AVG(fare_amount),2) AS avg_fare,
       ROUND(AVG(trip_distance),2) AS avg_trip_distance,
       ROUND(AVG(trip_duration_mins),2) AS avg_trip_duration,
       -- Credit card tips only — cash tips not electronically recorded
       -- tip / fare is used (not tip / total) which is the standard industry metric
       COALESCE(ROUND(AVG(CASE WHEN payment_type = 1 THEN tip_amount/NULLIF(fare_amount, 0) END) *100,2),0) AS tip_percentage,
       ROUND(COUNT(*) / SUM(COUNT(*)) OVER () * 100, 2) AS trip_share_pct,
       ROUND(SUM(total_amount), 2) AS total_revenue,
       ROUND(SUM(total_amount) / SUM(SUM(total_amount)) OVER () * 100, 2) AS revenue_share_pct
FROM taxi.taxi_silver.yellow_taxi_trips_cleaned
GROUP BY trip_type
) 
SELECT trip_type,
       trip_count, 
       avg_fare, 
       avg_trip_distance, 
       avg_trip_duration, 
       tip_percentage,
       trip_share_pct,
       total_revenue, 
       revenue_share_pct
FROM trip_type_stats
ORDER BY trip_type;

In [0]:
%sql
CREATE OR REPLACE TABLE taxi.taxi_gold.trip_type_in_depth_analysis AS 
WITH trip_type_analytics AS (
SELECT PULocationID,trip_type, COUNT(*) AS trip_count, 
       ROUND(AVG(fare_amount),2) AS avg_fare,
       ROUND(AVG(trip_distance),2) AS avg_trip_distance,
       ROUND(AVG(trip_duration_mins),2) AS avg_trip_duration,
       COALESCE(ROUND(AVG(CASE WHEN payment_type = 1 THEN tip_amount/NULLIF(fare_amount, 0) END) *100,2),0) AS tip_percentage
FROM taxi.taxi_silver.yellow_taxi_trips_cleaned
GROUP BY PULocationID,trip_type
),
top_airport_pickups AS (
SELECT b.Zone AS pickup_zone, b.Borough AS pickup_borough,
       a.trip_type,
       a.trip_count, 
       a.avg_fare, 
       a.avg_trip_distance, 
       a.avg_trip_duration, 
       a.tip_percentage,
       DENSE_RANK() OVER (PARTITION BY a.trip_type ORDER BY a.trip_count DESC) AS pickup_zones_rank
FROM trip_type_analytics AS a
JOIN taxi.taxi_bronze.yellow_taxi_lookup_raw_data AS b
ON a.PULocationID = b.LocationID
)
SELECT * 
FROM top_airport_pickups
WHERE trip_type = 'airport' AND pickup_zones_rank <=15
ORDER BY pickup_zones_rank;

**Congestion & Policy Analysis**

7. CBD congestion fee impact

Compare trips with cbd_congestion_fee > 0 vs those without. Did congestion pricing change trip behaviour — shorter trips, different zones, lower passenger counts? 

Also Track monthly trends.

In [0]:
%sql
CREATE OR REPLACE TABLE taxi.taxi_gold.congestion_impact_by_corridor AS 
WITH join_cte AS (
SELECT b.Borough AS pickup_borough, c.Borough AS dropoff_borough, a.congestion_zone_flag,
a.trip_distance, a.trip_duration_mins, a.passenger_count, a.fare_amount, a.total_amount
FROM taxi.taxi_silver.yellow_taxi_trips_cleaned AS a
JOIN taxi.taxi_bronze.yellow_taxi_lookup_raw_data AS b
ON a.PULocationID = b.LocationID
JOIN taxi.taxi_bronze.yellow_taxi_lookup_raw_data AS c
ON a.DOLocationID = c.LocationID
),
corridor_stats AS(
SELECT pickup_borough,
       dropoff_borough,
       congestion_zone_flag,
       COUNT(*) AS trip_count, 
       ROUND(AVG(trip_distance),2) AS avg_trip_distance,
       ROUND(AVG(trip_duration_mins),2) AS avg_trip_duration,
       ROUND(AVG(passenger_count),2) AS avg_passenger_count,
       ROUND(AVG(fare_amount), 2) AS avg_fare,
       ROUND(AVG(total_amount), 2) AS avg_total_amount
FROM join_cte
GROUP BY congestion_zone_flag, pickup_borough, dropoff_borough
) 
SELECT *, 
       DENSE_RANK() OVER(PARTITION BY congestion_zone_flag ORDER BY trip_count DESC) AS corridor_rank
FROM corridor_stats
ORDER BY congestion_zone_flag, corridor_rank;

In [0]:
%sql
CREATE OR REPLACE TABLE taxi.taxi_gold.congestion_impact_monthly  AS 
WITH monthly_info AS (SELECT pickup_month AS `month`,
       congestion_zone_flag,
       COUNT(*) AS trip_count,
       LAG(COUNT(*)) OVER (PARTITION BY congestion_zone_flag ORDER BY pickup_month) AS prev_month_trip_count,
       ROUND(AVG(trip_distance),2) AS avg_trip_distance,
       ROUND(AVG(trip_duration_mins),2) AS avg_trip_duration,
       ROUND(AVG(passenger_count),2) AS avg_passenger_count,
       ROUND(AVG(fare_amount), 2) AS avg_fare,
       ROUND(AVG(total_amount), 2) AS avg_total_amount
FROM taxi.taxi_silver.yellow_taxi_trips_cleaned
GROUP BY pickup_month, congestion_zone_flag
)
SELECT `month`,
        congestion_zone_flag,
        trip_count,
        prev_month_trip_count,
        ROUND((trip_count - prev_month_trip_count)/NULLIF(prev_month_trip_count,0) *100,2) AS trip_count_mom_pct,
        avg_trip_distance,
        avg_trip_duration,
        avg_passenger_count,
        avg_fare,
        avg_total_amount
FROM monthly_info
ORDER BY `month`, congestion_zone_flag;

8. Vendor performance comparison

Compare vendors across: total trips, average fare, average tip percentage, average trip distance, percentage of cash vs card payments. Is one vendor systematically different?

### Vendor Performance Analysis — Note on Vendor Coverage

The TLC data dictionary defines four active vendors for 2026:
- VendorID 1: Creative Mobile Technologies, LLC
- VendorID 2: Curb Mobility, LLC
- VendorID 6: Myle Technologies Inc
- VendorID 7: Helix

Only VendorID 1 and 2 appear in the Gold output. VendorID 6 (Myle) and
VendorID 7 (Helix) are present in Bronze but produce zero rows after
Silver cleaning. Every trip recorded under these vendors failed at least
one of the Silver quality filters:

- fare_amount >= 0
- total_amount >= 0
- trip_distance >= 0
- trip_duration_mins BETWEEN 1 AND 200
- passenger_count BETWEEN 1 AND 6
- tpep_pickup_datetime >= 2026-01-01

This suggests that Myle and Helix either use a different metering system
that records data in a format incompatible with TLC standards, or their
data for this period was corrupted at the source. They are retained in
Bronze for completeness and traceability, but are absent from all Silver
and Gold analyses.

In [0]:
%sql
CREATE OR REPLACE TABLE taxi.taxi_gold.vendor_performance AS
WITH vendor_stats AS (
SELECT vendorid_label,
       COUNT(*) AS total_trips,
       ROUND(AVG(trip_distance),2) AS avg_trip_distance,
       ROUND(AVG(fare_amount),2) AS avg_fare,
       ROUND(AVG(total_amount), 2) AS avg_total_amount,
       -- Credit card tips only — cash tips not electronically recorded
       ROUND(AVG(CASE WHEN payment_type =1 THEN tip_amount/NULLIF(fare_amount,0) *100 END),2) AS avg_tip_pct_cc,
       COUNT(CASE WHEN payment_type_label = "cash" THEN 1 END) AS cash_trip_count,
       COUNT(CASE WHEN payment_type_label = "credit_card" THEN 1 END) AS card_trip_count
FROM taxi.taxi_silver.yellow_taxi_trips_cleaned
GROUP BY vendorid_label
)
SELECT vendorid_label,
       total_trips,
       ROUND(total_trips / SUM(total_trips) OVER () * 100, 2) AS trip_share_pct,
       avg_trip_distance,
       avg_fare,
       avg_total_amount,
       avg_tip_pct_cc,
       cash_trip_count,
       card_trip_count,
       ROUND(cash_trip_count / total_trips * 100,2) AS pct_cash_trips,
       ROUND(card_trip_count / total_trips * 100,2) AS pct_card_trips
FROM vendor_stats
ORDER BY vendorid_label;

**Advanced patterns**

9. Monthly trend analysis with MoM growth

For each month in your dataset, calculate: total trips, total revenue, average fare, average tip rate, and average trip distance.Compute month-over-month growth rates for each metric. Which metrics grew fastest across the 4 months?

In [0]:
%sql
CREATE OR REPLACE TABLE taxi.taxi_gold.monthly_trend_analysis  AS 
WITH month_info AS (
SELECT pickup_month AS `month`,
       COUNT(*) AS trip_count,
       ROUND(AVG(trip_distance),2) AS avg_trip_distance,
       ROUND(AVG(trip_duration_mins),2) AS avg_trip_duration,
       ROUND(AVG(CASE WHEN payment_type = 1 THEN tip_amount END),2) AS avg_tip_amount,
       ROUND(AVG(CASE WHEN payment_type =1 THEN tip_amount/ NULLIF(fare_amount,0) *100 END),2) AS avg_tip_pct_cc,
       ROUND(AVG(fare_amount), 2) AS avg_fare,
       ROUND(SUM(total_amount), 2) AS total_revenue
FROM taxi.taxi_silver.yellow_taxi_trips_cleaned
GROUP BY pickup_month
),
prev_month_info AS (
SELECT `month`,
       LAG(`month`) OVER(ORDER BY `month`) AS prev_month,
       trip_count,
       LAG(trip_count) OVER(ORDER BY `month`) AS prev_month_trip_count,
       avg_trip_distance,
       LAG(avg_trip_distance) OVER(ORDER BY `month`) AS prev_month_avg_trip_distance,
       avg_trip_duration,
       LAG(avg_trip_duration) OVER(ORDER BY `month`) AS prev_month_avg_trip_duration,
       avg_tip_amount,
       avg_tip_pct_cc,
       LAG(avg_tip_pct_cc) OVER(ORDER BY `month`) AS prev_month_avg_tip_pct,
       avg_fare,
       LAG(avg_fare) OVER(ORDER BY `month`) AS prev_month_avg_fare,
       total_revenue,
       LAG(total_revenue) OVER(ORDER BY `month`) AS prev_month__total_revenue
FROM month_info
)
SELECT `month`,
       prev_month,
       trip_count,
       ROUND((trip_count - prev_month_trip_count) / prev_month_trip_count *100,2) AS trip_count_mom_pct,
       avg_trip_distance,
       ROUND( (avg_trip_distance - prev_month_avg_trip_distance) / prev_month_avg_trip_distance *100,2) AS avg_distance_mom_pct,
       avg_trip_duration,
       ROUND((avg_trip_duration - prev_month_avg_trip_duration) / prev_month_avg_trip_duration *100,2) AS avg_duration_mom_pct,
       avg_tip_amount,
       avg_tip_pct_cc,
       ROUND((avg_tip_pct_cc - prev_month_avg_tip_pct) / prev_month_avg_tip_pct *100,2) AS avg_tip_mom_pct,
       avg_fare,
       ROUND((avg_fare - prev_month_avg_fare) / prev_month_avg_fare *100,2) AS avg_fare_mom_pct,
       total_revenue,
       ROUND((total_revenue - prev_month__total_revenue) / prev_month__total_revenue *100,2) AS revenue_mom_pct
FROM prev_month_info
ORDER BY `month`;

## Summary

This notebook built a complete medallion architecture pipeline on NYC Yellow Taxi 2026 data:

**Bronze** — 2 Delta tables ingested from raw source files with no modifications  
**Silver** — 1 clean Delta table with ~17M validated trips and 10 derived columns  
**Gold** — 9 analytical Delta tables covering trip efficiency, fare breakdown, surge patterns, geographic analysis, congestion policy impact, vendor comparison, tipping behaviour regression, and monthly trends

All Gold tables feed directly into the Databricks dashboard.  